# External validation — reconstructed inter-state goods flows vs CFS 2017 and FAF5

The reconstructed intra-United-States bilateral flows carry an **estimated** state-by-state
structure (gravity model), absent from the WiNDC source. Here we confront that structure
with two independent observations of inter-state commodity movement:

* **CFS 2017** — Commodity Flow Survey public-use microdata (5.98 M shipments); domestic
  state-to-state value flows are obtained by summing `SHIPMT_VALUE × WGT_FACTOR` over
  `EXPORT_YN = N`.
* **FAF5.7.1 (State)** — Freight Analysis Framework, 2017 base year; domestic state O-D value
  flows are `value_2017` restricted to `trade_type = 1`.

Both observe only **physically shippable goods**, so the reconstructed flows are restricted
to the goods-producing sectors (agriculture, mining, oil & gas, all manufacturing, primary
metals), summed over intermediate **and** final-demand deliveries. Levels are not comparable
across sources (survey shipment value vs input-output value), so the comparison is on
**structure** (bilateral shares), on the 2550 ordered inter-state pairs of 2017.

Four metrics: (1) state-pair correlation of bilateral value flows; (2) a flow-weighted
structural error; (3) recovered export/import partner shares; (4) top-partner recovery.
CFS-vs-FAF is reported as a reference for how well the two observations agree with each other.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))   # `paths` without an install
from paths import ROOT

FIG  = ROOT/"figures"; FIG.mkdir(parents=True, exist_ok=True)
CFS_CSV = ROOT/"data/raw/CFS/CFS 2017 PUF CSV.csv"
FAF_CSV = ROOT/"data/raw/FAF5/FAF5.7.1_State.csv"
HARM    = ROOT/"data/interim/IOT/IOT_USA/grav_fric_v3.1_RAS_harmonized/IOT_2017_harmonized.npz"

FIPS={'01':'AL','02':'AK','04':'AZ','05':'AR','06':'CA','08':'CO','09':'CT','10':'DE',
'11':'DC','12':'FL','13':'GA','15':'HI','16':'ID','17':'IL','18':'IN','19':'IA','20':'KS',
'21':'KY','22':'LA','23':'ME','24':'MD','25':'MA','26':'MI','27':'MN','28':'MS','29':'MO',
'30':'MT','31':'NE','32':'NV','33':'NH','34':'NJ','35':'NM','36':'NY','37':'NC','38':'ND',
'39':'OH','40':'OK','41':'OR','42':'PA','44':'RI','45':'SC','46':'SD','47':'TN','48':'TX',
'49':'UT','50':'VT','51':'VA','53':'WA','54':'WV','55':'WI','56':'WY'}
print("ROOT:", ROOT)

### Reconstructed goods flows (delivered intra-US block, 19 goods sectors)

In [ ]:
d=np.load(HARM, allow_pickle=True)
secs=[str(s) for s in d["proposed_sectors"]]; STATES=[str(r) for r in d["regions"]]
ns,nr=len(secs),len(STATES)
GOODS=[i for i,s in enumerate(secs) if s.startswith("Manufacture") or s in
   {"Agriculture, hunting, forestry, fishing and related","Mining, except oil & gas",
    "Oil and gas extraction","primary metals"}]
Z4=d["Z"].reshape(nr,ns,nr,ns); F3=d["F"].reshape(nr,ns,nr,-1)
M_recon=(Z4[:,GOODS,:,:].sum((1,3)) + F3[:,GOODS,:,:].sum((1,3)))   # (51,51) origin->dest, Bn$
IDX={a:i for i,a in enumerate(STATES)}
print(f"{len(GOODS)} goods sectors; recon inter-state goods flow = "
      f"{(M_recon.sum()-np.trace(M_recon))/1e3:.2f} tn$ (intra-state {np.trace(M_recon)/M_recon.sum():.1%})")

### CFS 2017 and FAF5 domestic goods flows (chunked, memory-safe)

In [ ]:
def mat_from_pairs(dct):
    M=np.zeros((nr,nr))
    for (o,dd),v in dct.items():
        if o in IDX and dd in IDX: M[IDX[o],IDX[dd]]=v
    return M

# CFS: weighted domestic shipments
agg={}
for ch in pd.read_csv(CFS_CSV,
        usecols=["ORIG_STATE","DEST_STATE","EXPORT_YN","SHIPMT_VALUE","WGT_FACTOR"],
        dtype={"ORIG_STATE":str,"DEST_STATE":str,"EXPORT_YN":str}, chunksize=1_500_000):
    ch=ch[ch["EXPORT_YN"]=="N"].copy()
    ch["o"]=ch["ORIG_STATE"].map(FIPS); ch["d"]=ch["DEST_STATE"].map(FIPS)
    ch=ch.dropna(subset=["o","d"]); ch["v"]=ch["SHIPMT_VALUE"]*ch["WGT_FACTOR"]
    for k,v in ch.groupby(["o","d"])["v"].sum().items(): agg[k]=agg.get(k,0)+v
M_cfs=mat_from_pairs(agg)

# FAF: domestic (trade_type=1) value_2017
aggf={}
for ch in pd.read_csv(FAF_CSV,
        usecols=["dms_origst","dms_destst","trade_type","value_2017"],
        dtype={"dms_origst":str,"dms_destst":str,"trade_type":str}, chunksize=1_500_000):
    ch=ch[ch["trade_type"]=="1"].copy()
    ch["o"]=ch["dms_origst"].map(FIPS); ch["d"]=ch["dms_destst"].map(FIPS)
    ch=ch.dropna(subset=["o","d"])
    for k,v in ch.groupby(["o","d"])["value_2017"].sum().items(): aggf[k]=aggf.get(k,0)+v
M_faf=mat_from_pairs(aggf)
print(f"CFS domestic goods = {(M_cfs.sum()-np.trace(M_cfs))/1e12:.2f} tn$ inter-state;  "
      f"FAF = {(M_faf.sum()-np.trace(M_faf))/1e6:.2f} tn$ inter-state")

### Metrics

In [ ]:
off=~np.eye(nr,dtype=bool)
def shares(M):
    s=M.copy(); np.fill_diagonal(s,0.0); return s/s.sum()
def logcorr(A,B):
    a,b=A[off],B[off]; m=(a>0)&(b>0)
    return stats.pearsonr(np.log(a[m]),np.log(b[m]))[0], m.sum()
def spear(A,B):
    a,b=A[off],B[off]; m=(a>0)&(b>0)
    return stats.spearmanr(a[m],b[m])[0]
def struct_L1(A,B): return np.abs(shares(A)[off]-shares(B)[off]).sum()   # 0..2; /2 = mass misallocated
def rownorm(M):
    X=M.copy(); np.fill_diagonal(X,0.0); r=X.sum(1,keepdims=True); r[r==0]=1; return X/r
def colnorm(M):
    X=M.copy(); np.fill_diagonal(X,0.0); c=X.sum(0,keepdims=True); c[c==0]=1; return X/c
def partner_L1(A,B,axis="row"):
    RA,RB=(rownorm(A),rownorm(B)) if axis=="row" else (colnorm(A),colnorm(B))
    d=np.abs(RA-RB).sum(1 if axis=="row" else 0); return d[d>0].mean() if (d>0).any() else 0.0
def top_recovery(A,B,axis="row"):
    XA=A.copy(); XB=B.copy(); np.fill_diagonal(XA,-1); np.fill_diagonal(XB,-1)
    if axis=="row": return (XA.argmax(1)==XB.argmax(1)).mean()
    return (XA.argmax(0)==XB.argmax(0)).mean()

rows=[]
for nm,Mo in [("recon vs CFS",M_cfs),("recon vs FAF",M_faf),("CFS vs FAF (ref.)",M_faf)]:
    A = M_recon if nm!="CFS vs FAF (ref.)" else M_cfs
    r,n=logcorr(A,Mo)
    rows.append(dict(comparison=nm, pearson_log=r, spearman=spear(A,Mo),
                     struct_L1=struct_L1(A,Mo), exp_partner_L1=partner_L1(A,Mo,"row"),
                     top_exp_recovery=top_recovery(A,Mo,"row"),
                     top_imp_recovery=top_recovery(A,Mo,"col"), n_pairs=n))
T=pd.DataFrame(rows)
print(T.to_string(index=False, formatters={
  "pearson_log":"{:.3f}".format,"spearman":"{:.3f}".format,"struct_L1":"{:.3f}".format,
  "exp_partner_L1":"{:.3f}".format,"top_exp_recovery":"{:.1%}".format,"top_imp_recovery":"{:.1%}".format}))
T.to_csv(FIG/"cfs_faf_metrics_2017.csv", index=False)

# California over-concentration (the Nevada example of the paper)
def ca_exp_share(M):
    X=M.copy(); np.fill_diagonal(X,0.0); return X[:,IDX["CA"]]/X.sum(1)
print("\nExport share to California (uniform-decay over-concentration):")
for nm,M in [("recon",M_recon),("CFS",M_cfs),("FAF",M_faf)]:
    cs=ca_exp_share(M)
    print(f"  {nm:6s} mean={np.nanmean(cs):.1%}   Nevada->CA={cs[IDX['NV']]:.1%}")

### Figure — `fig_cfs_faf_validation.png`

In [ ]:
def scfmt(ax,x,y,lab,ttl):
    m=(x>0)&(y>0); x,y=x[m],y[m]
    lo,hi=min(x.min(),y.min()),max(x.max(),y.max())
    ax.plot([lo,hi],[lo,hi],"--",color="#3577a8",lw=1.2,zorder=1,label="y = x")
    ax.scatter(x,y,s=10,c="#bd5e2c",alpha=.55,edgecolor="none",zorder=2)
    ax.set_xscale("log");ax.set_yscale("log");ax.set_aspect("equal","datalim")
    r=stats.pearsonr(np.log(x),np.log(y))[0]
    ax.set_title(f"{ttl}   r={r:.3f}",fontsize=10,fontweight="bold")
    ax.set_xlabel(lab[0],fontsize=8);ax.set_ylabel(lab[1],fontsize=8)
    ax.grid(ls=":",alpha=.4);ax.legend(frameon=False,fontsize=8,loc="lower right")

fig,ax=plt.subplots(2,2,figsize=(13,12))
scfmt(ax[0,0],M_cfs[off],M_recon[off],("CFS shipment value","reconstructed IO value"),
      "State-pair goods flows: recon vs CFS")
scfmt(ax[0,1],M_faf[off],M_recon[off],("FAF value 2017","reconstructed IO value"),
      "State-pair goods flows: recon vs FAF")
# (c) export-partner shares recon vs CFS
Rc,Rr=rownorm(M_cfs),rownorm(M_recon)
scfmt(ax[1,0],Rc[off],Rr[off],("CFS export-partner share","recon export-partner share"),
      "Export-partner shares: recon vs CFS")
# (d) export share to California, recon vs CFS
cs_c,cs_r=ca_exp_share(M_cfs),ca_exp_share(M_recon)
a=ax[1,1]
lo,hi=0,max(cs_c.max(),cs_r.max())*1.05
a.plot([lo,hi],[lo,hi],"--",color="#3577a8",lw=1.2,label="y = x")
a.fill_between([lo,hi],[lo,hi],[hi,hi],color="#bd5e2c",alpha=.06)
a.scatter(cs_c,cs_r,s=22,c="#bd5e2c",alpha=.7,edgecolor="none")
nv=IDX["NV"]; a.scatter(cs_c[nv],cs_r[nv],s=80,facecolor="none",edgecolor="k",lw=1.5)
a.annotate("NV",(cs_c[nv],cs_r[nv]),textcoords="offset points",xytext=(6,4),fontsize=9)
a.set_title("Export share to California — over-concentration",fontsize=10,fontweight="bold")
a.set_xlabel("CFS share to CA",fontsize=8);a.set_ylabel("recon share to CA",fontsize=8)
a.grid(ls=":",alpha=.4);a.legend(frameon=False,fontsize=8,loc="upper left")
fig.suptitle("External validation of reconstructed inter-state goods flows, 2017",
             fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig(FIG/"fig_cfs_faf_validation.png",dpi=160,bbox_inches="tight")
plt.show()
print("saved", FIG/"fig_cfs_faf_validation.png")